# Faz 2 — Vaka Bazlı Split + Ön İşleme (Fold 0)

**Amaç:** Faz 1'de üretilen `manifest.csv`'yi kullanarak:
1. Hold-out (~%15) ayrılır
2. Kalan vakalar GroupKFold (k=5) ile bölünür → sadece **fold 0** ile çalışılır
3. Fold 0 için YOLO formatında (PNG + .txt) eğitim/doğrulama veri seti üretilir

**Önemli tasarım kararı:** Aynı vakanın kesitleri ASLA farklı fold'lara düşmez (GroupKFold, group=Case Number). Bu, data leakage'i önler ve hasta-düzeyi değerlendirme için zorunludur.

**Süre tahmini (fold 0, lokal):** Split <1 dk, YOLO export ~10-30 dk (annotasyonlu kesit sayısına göre).

## 1. Ortam (Faz 1 ile aynı)

In [ ]:
import os, sys
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()

DATA_ROOT = Path(os.environ.get("TR_ABDOMEN_BASE", r"D:/makale-pdf/Proje/abdomen"))
PROJE     = Path(os.environ.get("TR_ABDOMEN_PROJE", r"D:/makale-pdf/Proje"))
CODE      = Path(os.environ.get("TR_ABDOMEN_CODE",  r"D:/makale-pdf/Proje/code"))
EGITIM_DIR = DATA_ROOT / "Eğitim Verisi" / "Eg╠åitim Verisi.zip"

os.environ["ABDOMEN_PROJECT_ROOT"] = str(PROJE)
os.environ["ABDOMEN_DATA_ROOT"]    = str(DATA_ROOT)
os.environ["ABDOMEN_TRAIN_DIR"]    = str(EGITIM_DIR)
os.environ["ABDOMEN_TEST_DIR"]     = str(DATA_ROOT / "Yarışma Veri Seti")
os.environ["ABDOMEN_BILGI_XLSX"]   = str(DATA_ROOT / "Bilgi.xlsx")
os.environ["ABDOMEN_OUT_DIR"]      = str(CODE / "outputs")

sys.path.insert(0, str(CODE))

from src.config import SPLIT_DIR, DET_DATA_DIR, SUPER_CLASSES, DEFAULT_SPLIT
print("SPLIT_DIR  :", SPLIT_DIR)
print("DET_DATA_DIR:", DET_DATA_DIR)
print("Fold ayarı : k =", DEFAULT_SPLIT.n_splits, ", hold-out=", DEFAULT_SPLIT.holdout_frac)

## 2. GroupKFold Split (5 fold + hold-out)

In [ ]:
from src.splits import make_splits, load_fold, load_holdout

# Bu çağrı outputs/splits/ altına şunları yazar:
#   - holdout.csv
#   - splits.csv (her vakanın hangi fold'da olduğu)
#   - fold{N}_train.csv, fold{N}_val.csv (N=0..4)
paths = make_splits()
for k, v in paths.items():
    print(f"  {k}: {v}")

In [ ]:
# Hold-out + fold 0 doğrulama
import pandas as pd

holdout_cases = load_holdout()
fold0_train   = load_fold(fold=0, split="train")
fold0_val     = load_fold(fold=0, split="val")

print(f"Hold-out vaka sayısı : {len(holdout_cases)}")
print(f"Fold 0 - train vakalar: {len(fold0_train)}")
print(f"Fold 0 - val vakalar  : {len(fold0_val)}")

# Kesişim kontrolü (olmamalı)
overlap = set(fold0_train) & set(fold0_val)
print(f"\nTrain∩Val kesişim: {len(overlap)} (0 olmalı)")
overlap_h = set(fold0_train + fold0_val) & set(holdout_cases)
print(f"Fold0∪Holdout kesişim: {len(overlap_h)} (0 olmalı)")

In [ ]:
# Fold 0'da üst sınıf dağılımı
manifest = pd.read_csv(SPLIT_DIR / "manifest.csv")
manifest['super_labels'] = manifest['super_labels'].fillna('').astype(str)

def count_classes(cases):
    sub = manifest[manifest['case'].isin(cases)]
    cnt = {s: 0 for s in range(len(SUPER_CLASSES))}
    for v in sub['super_labels']:
        if not v: continue
        for s in v.split(';'):
            if s.strip(): cnt[int(s)] += 1
    return cnt

import pandas as pd
rows = []
for name, cases in [("train", fold0_train), ("val", fold0_val), ("holdout", holdout_cases)]:
    cnt = count_classes(cases)
    rows.append([name, len(cases)] + [cnt[i] for i in range(len(SUPER_CLASSES))])
dist = pd.DataFrame(rows, columns=["split", "n_case"] + SUPER_CLASSES)
dist

## 3. Fold 0 için YOLO Veri Seti Üretimi

`detection.export_yolo_dataset(fold=0)`:
- Annotasyonlu (BBox'lu) kesitleri PNG (3-kanal HU pencereleme) olarak yazar
- Her PNG için YOLO formatında `.txt` etiketi (`class cx cy w h` normalize) üretir
- `dataset.yaml` dosyası ile Ultralytics'in beklediği yapıyı kurar

**Çıktı yapısı:**
```
outputs/det_data/fold0/
├── images/train/*.png
├── images/val/*.png
├── labels/train/*.txt
├── labels/val/*.txt
└── dataset.yaml
```

In [ ]:
from src.detection import export_yolo_dataset

fold = 0
fold_dir = export_yolo_dataset(fold=fold, include_val_negatives=False)
print("YOLO veri seti hazır:", fold_dir)

In [ ]:
# Çıktı sağlaması
train_imgs = list((fold_dir / "images" / "train").glob("*.png"))
val_imgs   = list((fold_dir / "images" / "val").glob("*.png"))
train_lbls = list((fold_dir / "labels" / "train").glob("*.txt"))
val_lbls   = list((fold_dir / "labels" / "val").glob("*.txt"))

print(f"Train: {len(train_imgs):,} PNG, {len(train_lbls):,} label")
print(f"Val  : {len(val_imgs):,} PNG, {len(val_lbls):,} label")
print(f"\nÖrnek etiket dosyası ({train_lbls[0].name}):")
print(train_lbls[0].read_text()[:300])

print(f"\ndataset.yaml içeriği:")
print((fold_dir / "dataset.yaml").read_text())

## 4. Görsel Sağlama — Rastgele 4 Eğitim Kesiti + BBox'lar

In [ ]:
import random, cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
import numpy as np

random.seed(7)
samples = random.sample(train_imgs, 4)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, img_path in zip(axes, samples):
    img = np.array(Image.open(img_path))
    ax.imshow(img)
    # Etiketi oku
    lbl_path = fold_dir / "labels" / "train" / (img_path.stem + ".txt")
    if lbl_path.exists():
        H, W = img.shape[:2]
        for line in lbl_path.read_text().strip().splitlines():
            parts = line.split()
            cls = int(parts[0]); cx, cy, w, h = [float(x) for x in parts[1:5]]
            x1 = (cx - w/2) * W; y1 = (cy - h/2) * H
            ax.add_patch(mpatches.Rectangle((x1, y1), w*W, h*H,
                                            linewidth=2, edgecolor='red', facecolor='none'))
            ax.text(x1, y1-3, SUPER_CLASSES[cls][:15],
                    color='red', fontsize=8, backgroundcolor='white')
    ax.set_title(img_path.name, fontsize=8)
    ax.axis('off')
plt.tight_layout(); plt.show()

## 5. Faz 2 Çıktı Özeti

| Çıktı | Yol |
|---|---|
| Splits | `outputs/splits/{holdout,splits,fold0_train,fold0_val}.csv` |
| YOLO data | `outputs/det_data/fold0/{images,labels}/{train,val}/` |
| dataset.yaml | `outputs/det_data/fold0/dataset.yaml` |

**Sonraki adım:** Faz 3 → `Faz3_YOLO_Baseline_1fold.ipynb`